# 10 — RAG: clean the text and split it into chunks

**Purpose.** Turn the extracted text of the 183 corpus documents (notebook 09) into clean, citable **chunks**, one record per passage, ready for embedding. Contract: notebook 08.

- **Inputs:** newest `data/manifests/rag_corpus_*.json` and the text files it lists under `data/raw/rag/text/`; `configs/rag.yaml` (`chunking`).
- **Outputs:**
  - `data/processed/rag/chunks.jsonl`: one chunk per line (git-ignored, rebuilt by this notebook)
  - `data/manifests/rag_chunks_<timestamp>.json`: parameters, counts, hashes, per-document report (committed)
- **Next:** notebook 11 embeds the chunks with `bge-m3` and builds the hybrid index.

Needs no GPU. It downloads only the small `bge-m3` tokenizer (used to measure chunk length in tokens).

### For the evaluator

- **Every chunk carries what a citation needs:** `doc_id`, `chunk_id`, `title`, `landing_url`, `language`, `published_effective` (use this, not `published`, for the time cutoff), `heading`, `text`.
- **`text` is what the user sees and what a citation quotes. `embed_text` is what gets embedded** (`text` plus the document title and section heading, so a passage such as "up 4%" is not meaningless on its own).
- **Chunk ids are stable** for the same corpus and parameters (`<doc_id>#<index>`). Rebuild by running this notebook top to bottom; the manifest records the input and output hashes.
- **Tables are deliberately not indexed.** See "What is dropped and why". Questions whose answer only exists in a bulletin table will have no supporting chunk, so treat them as *unanswerable*, not as retrieval failures.
- **Coverage per document is in the manifest** (`documents[].chunks`), so expected-source questions can be checked against what was actually indexed.

In [ ]:
# Mount Drive on Colab; skipped automatically when running locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
except ImportError:
    pass

In [ ]:
# Only needed on a fresh runtime; the project requirements already list these.
# !pip install -q tokenizers pyyaml pandas

In [ ]:
import os, re, json, hashlib, statistics, datetime as dt, importlib.metadata
from pathlib import Path
import yaml
import pandas as pd

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "rag.yaml").is_file():
            return cand
    return p

REPO = _find_repo()
MAN = REPO / "data" / "manifests"
TEXT_DIR = REPO / "data" / "raw" / "rag" / "text"
OUT_DIR = REPO / "data" / "processed" / "rag"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))
CHUNK_CFG = CFG["chunking"]
# EVALUATOR: these are the chunking parameters. Change them in configs/rag.yaml, not here,
# so that every run records the same values in its manifest.
MAX_TOKENS = int(CHUNK_CFG["chunk_size_tokens"])
OVERLAP_TOKENS = int(CHUNK_CFG["chunk_overlap_tokens"])
MIN_TOKENS = int(CHUNK_CFG["min_chunk_tokens"])
TOKENIZER_ID = CHUNK_CFG["tokenizer"]
print("repo:", REPO)
print("chunking:", {"max": MAX_TOKENS, "overlap": OVERLAP_TOKENS, "min": MIN_TOKENS, "tokenizer": TOKENIZER_ID})

## Inputs check

Pin the corpus manifest, so the chunks say exactly which corpus they came from. `CORPUS_MANIFEST = None` takes the newest one; set a file name to reproduce an older run. The check stops here, with a clear message, if a text file is missing.

In [ ]:
CORPUS_MANIFEST = None   # e.g. "rag_corpus_20260921T175041Z.json"

if CORPUS_MANIFEST:
    manifest_path = MAN / CORPUS_MANIFEST
else:
    found = sorted(MAN.glob("rag_corpus_*.json"))
    assert found, "No rag_corpus_*.json manifest found. Run notebook 09 first."
    manifest_path = found[-1]

corpus = json.loads(manifest_path.read_text())
docs = pd.DataFrame(corpus["documents"])
missing = [d for d in docs.doc_id if not (TEXT_DIR / f"{d}.txt").is_file()]
assert not missing, f"{len(missing)} text files missing (e.g. {missing[:3]}). Run notebook 09 with DRY_RUN = False."
print("corpus manifest:", manifest_path.name)
print("documents:", len(docs), "| by source:", docs.source_id.value_counts().to_dict(),
      "| by language:", docs.language.value_counts().to_dict())

## Cleaning

The PDF text is usable but messy (checked on real bulletins). The cleaner fixes what can be fixed and drops what cannot be used.

| Problem in the extracted text | Example | What the cleaner does |
|---|---|---|
| Non-breaking spaces everywhere | `työttömiä\xa0työnhakijoita` | Turn into normal spaces |
| A soft-hyphen character used as the real hyphen and as the minus sign | `Varsinais\xadSuomi`, `(\xad11 500)` | Turn into `-` |
| Sentences hard-wrapped at about 60 characters | `…oli elokuun` / `lopussa yhteensä…` | Join wrapped lines back into paragraphs |
| Running page header on each page | `Elokuu 2024` | Remove |
| Chart and map labels, one value per line | `Uusimaa` / `10,8` / `50 000` | Drop |
| Table rows | `Uusimaa 98 059 104 946 85 760 …` | Drop |
| Contacts, links, referencing instructions | `Lisätietoja:` / `Links:` sections | Drop |

### What is dropped and why

Table and chart numbers are dropped on purpose. Two reasons: (1) flattened by the PDF, a table row has lost its column headers, so it cannot be read without them; (2) the contract says **numbers come from the PxWeb data and the forecast, never from retrieval**. What stays is prose: the "Trends" bullets and the narrative under each section, which is where the bulletin explains *what happened*.

**Trade-off:** regional figures that appear only in a table are not searchable. That is a deliberate limit of version 1, not a bug.

In [ ]:
TERMINAL = ".!?:;"
MONTH_RE = (r"(?:tammikuu|helmikuu|maaliskuu|huhtikuu|toukokuu|kesäkuu|heinäkuu|elokuu|syyskuu|lokakuu|"
            r"marraskuu|joulukuu|january|february|march|april|may|june|july|august|september|october|november|december)")

# Section headings that recur in every bulletin. They label chunks and tell us where a section starts.
HEADINGS = {
    "fi": ["Trendit:", "Kaikki työnhakijat", "Työttömät työnhakijat", "Työttömät työnhakijat alueittain",
           "Pitkäaikaistyöttömyys", "Palvelut", "Työllistäminen", "Avoimet työpaikat",
           "Työvoimakoulutus, valmennukset ja omaehtoinen opiskelu",
           "Vuorotteluvapaa, kuntouttava työtoiminta ja kokeilut", "Kuntouttava työtoiminta ja kokeilut",
           "Lisätietoja:"],
    "en": ["Trends:", "All jobseekers", "Unemployed jobseekers", "Unemployed jobseekers by region",
           "Long-term unemployment", "Services", "Employment", "Jobs vacant",
           "Labour market training, coaching and self-motivated studies",
           "Rehabilitative work activities and trials", "Further information:", "Links:"],
}
DROP_HEADINGS = {"Lisätietoja:", "Links:", "Further information:"}   # contacts, links, referencing instructions

def normalise_line(line):
    line = line.replace("\xa0", " ").replace("​", "").replace("﻿", "").replace("­", "-")
    return re.sub(r"[ \t]+", " ", line).strip()

def is_running_header(line):
    return bool(re.fullmatch(rf"{MONTH_RE}\s+20\d\d|20\d\d\s+{MONTH_RE}", line, re.I))

def is_word(token):
    core = token.strip(".,;:!?()[]%\"'")
    return len(core) >= 3 and sum(c.isalpha() for c in core) / len(core) >= 0.8

def is_upper_line(line):
    letters = [c for c in line if c.isalpha()]
    return len(letters) >= 6 and sum(c.isupper() for c in letters) / len(letters) >= 0.8

def line_kind(line):
    """'blank', 'text' (part of a sentence) or 'other' (label, number, table row, caption)."""
    words = line.split()
    if not words:
        return "blank"
    if is_upper_line(line):
        return "other"                                   # captions and appendix titles
    word_ratio = sum(is_word(w) for w in words) / len(words)
    unique_ratio = len({w.lower() for w in words}) / len(words)   # table headers repeat words
    if len(words) >= 5 and word_ratio >= 0.4 and unique_ratio >= 0.6:
        return "text"
    if len(words) >= 3 and word_ratio >= 0.6 and line[-1] in TERMINAL:
        return "text"                                    # a short sentence
    return "other"

def is_prose(paragraph, heading):
    words = paragraph.split()
    if len(words) < 6:
        return False
    if (heading or "").endswith(":"):
        return True                                      # trend bullets have no full stop
    if paragraph[-1] not in ".!?" and len(words) < 40:
        return False                                     # short table rows and captions end in numbers or labels
    alpha = [w for w in words if is_word(w)]
    lower = sum(w[:1].islower() for w in alpha)
    numeric = sum(any(c.isdigit() for c in w) and not is_word(w) for w in words)
    return len(alpha) >= 5 and lower / len(alpha) >= 0.55 and numeric / len(words) <= 0.35

def clean_paragraphs(raw, lang):
    """Raw extracted text -> [{'heading', 'text'}] prose paragraphs in reading order."""
    headings = {h.lower(): h for h in HEADINGS[lang]}
    lines = [normalise_line(l) for l in raw.split("\n")]
    lines = [l for l in lines if not is_running_header(l)]
    kinds = [line_kind(l) for l in lines]
    text_lengths = [len(l) for l, k in zip(lines, kinds) if k == "text"]
    width = statistics.median(text_lengths) if text_lengths else 60   # typical wrapped-line width

    paragraphs, current, heading = [], [], None
    def flush():
        nonlocal current
        if current:
            text = " ".join(current).strip()
            if heading not in DROP_HEADINGS and is_prose(text, heading):
                paragraphs.append({"heading": heading, "text": text})
            current = []

    prev_kind, prev = None, ""
    for line, kind in zip(lines, kinds):
        if line.lower() in headings:
            flush()
            heading = headings[line.lower()]
            prev_kind, prev = "heading", line
            continue
        if kind == "blank":
            flush()
            prev_kind = "blank"
            continue
        if kind == "text":
            # A short last line that ends a sentence closes the paragraph.
            if current and prev_kind in ("text", "tail") and prev[-1:] in TERMINAL and len(prev) < 0.75 * width:
                flush()
            if prev_kind not in ("text", "tail"):
                flush()
            current.append(line)
            prev_kind = "text"
        elif (prev_kind == "text" and prev[-1:] not in TERMINAL and len(line.split()) <= 4
              and (line[:1].islower() or line[:1].isdigit() or len(prev) >= 0.85 * width)):
            current.append(line)                         # wrapped tail of the previous line (at most one)
            prev_kind = "tail"
        else:
            flush()
            prev_kind = "other"
        prev = line
    flush()
    return paragraphs

### Cleaning check on real documents

One Finnish and one English bulletin, before and after. Look at the vacancy section: it should read as complete sentences.

In [ ]:
def show(doc_id, heading_match, n_raw=420, n_clean=700):
    row = docs.set_index("doc_id").loc[doc_id]
    raw = (TEXT_DIR / f"{doc_id}.txt").read_text()
    paras = clean_paragraphs(raw, row.language)
    view = raw.replace("\xa0", " ").replace("\u00ad", "-")      # so the heading is found
    kept = sum(len(p["text"]) for p in paras)
    print(f"=== {doc_id} ({row.language}): {len(raw):,} raw chars -> {kept:,} kept ({100 * kept // len(raw)}%), {len(paras)} paragraphs")
    hit = re.search(rf"^{re.escape(heading_match)}\s*$", view, re.I | re.M)   # the heading as a whole line
    i = hit.start() if hit else 0
    print("--- RAW around the vacancy section:")
    print(repr(view[i:i + n_raw]))
    print("--- CLEAN, same section:")
    for p in paras:
        if p["heading"] == heading_match:
            print(p["text"][:n_clean], "\n")
            break

show("tem_bulletin_2024_08", "Avoimet työpaikat")
show("keha_bulletin_20260821118810", "Jobs vacant")

## Cleaning report over the whole corpus

Checks that cleaning worked on every document, not just the two above. A document with **no vacancy paragraph** is listed: it may be genuine (some bulletins show vacancies only in a chart) or a cleaning problem worth a look.

In [ ]:
VACANCY_PATTERN = {"fi": r"avoim|työpaikk", "en": r"vacanc|job openings"}

cleaned, report = {}, []
for d in docs.itertuples():
    raw = (TEXT_DIR / f"{d.doc_id}.txt").read_text()
    paras = clean_paragraphs(raw, d.language)
    cleaned[d.doc_id] = paras
    kept = sum(len(p["text"]) for p in paras)
    vacancy = [p for p in paras if re.search(VACANCY_PATTERN[d.language], p["text"], re.I) and re.search(r"\d", p["text"])]
    report.append({"doc_id": d.doc_id, "source_id": d.source_id, "language": d.language,
                   "raw_chars": len(raw), "kept_chars": kept, "paragraphs": len(paras),
                   "vacancy_paragraphs": len(vacancy)})
report = pd.DataFrame(report)

print(report.groupby("source_id").agg(
    docs=("doc_id", "size"),
    kept_pct=("kept_chars", lambda s: round(100 * s.sum() / report.loc[s.index, "raw_chars"].sum())),
    paragraphs_median=("paragraphs", "median"), paragraphs_min=("paragraphs", "min"),
    vacancy_paragraphs_median=("vacancy_paragraphs", "median")))
no_vacancy = report[report.vacancy_paragraphs == 0]
print("\ndocuments with no vacancy paragraph:", len(no_vacancy), no_vacancy.doc_id.tolist())
assert report.paragraphs.min() > 0, "a document lost all its text"

## Chunking

Version 1 settings come from `configs/rag.yaml`: **512 tokens per chunk, 64 tokens of overlap**, sizes measured with the `bge-m3` tokenizer (a Finnish word is about 2 tokens).

How chunks are built, in order:

1. **Keep paragraphs whole when possible.** Paragraphs are packed into a chunk until the next one would not fit; then a new chunk starts. This is the "recursive" idea: try the biggest natural boundary first.
2. **Only if a single paragraph is too long, split it by sentences,** and only if one sentence is too long, split it by words.
3. **Never cross a section.** A new heading starts a new chunk (unless the current chunk is under the minimum size).
4. **Overlap.** A new chunk begins with the last sentence(s) of the previous one, up to 64 tokens, so a sentence cut at a boundary can still be found from either side. Overlap is not carried across a heading.
5. **No tiny leftovers.** A final chunk under 40 tokens is merged into the previous one if it fits.
6. **Add context for embedding.** `embed_text` = document title + section heading + `text`; `text` itself stays untouched for citations.

In [ ]:
from tokenizers import Tokenizer

TOK = Tokenizer.from_pretrained(TOKENIZER_ID)
def count_tokens(texts):
    """Token counts (no special tokens) for a list of strings."""
    return [len(e.ids) for e in TOK.encode_batch(texts, add_special_tokens=False)]

SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-ZÅÄÖ0-9(\-])")

def to_units(paragraph_text):
    """Sentences of a paragraph with their token counts; an over-long sentence is cut into word windows."""
    sentences = SENTENCE_SPLIT.split(paragraph_text)
    units = []
    for sentence, n in zip(sentences, count_tokens(sentences)):
        if n <= MAX_TOKENS:
            units.append((sentence, n))
            continue
        words, window = sentence.split(), []
        for word in words:
            window.append(word)
            if count_tokens([" ".join(window)])[0] > MAX_TOKENS:
                window.pop()
                units.append((" ".join(window), count_tokens([" ".join(window)])[0]))
                window = [word]
        if window:
            units.append((" ".join(window), count_tokens([" ".join(window)])[0]))
    return units

def chunk_paragraphs(paragraphs):
    """[{'heading', 'text'}] -> [{'heading', 'text'}] chunks (packing, overlap, no tiny leftovers)."""
    chunks, current = [], None     # current = {'heading', 'units': [(sentence, tokens)], 'tokens'}

    def overlap_tail(units):
        tail, total = [], 0
        for unit in reversed(units[:-1]):     # never carry the whole chunk
            if total + unit[1] > OVERLAP_TOKENS:
                break
            tail.insert(0, unit)
            total += unit[1]
        return tail

    def start(heading, carried=()):
        return {"heading": heading, "units": list(carried), "tokens": sum(u[1] for u in carried)}

    def emit():
        if current and current["units"]:
            chunks.append(current)

    for para in paragraphs:
        units = to_units(para["text"])
        para_tokens = sum(u[1] for u in units)
        new_section = current is not None and para["heading"] != current["heading"]
        if current is None:
            current = start(para["heading"])
        elif new_section and current["tokens"] >= MIN_TOKENS:
            emit(); current = start(para["heading"])              # never cross a section
        elif current["tokens"] + para_tokens > MAX_TOKENS and para_tokens <= MAX_TOKENS:
            carried = overlap_tail(current["units"]) if not new_section else []
            emit(); current = start(para["heading"], carried)     # whole paragraph goes to the next chunk
        if current["tokens"] + para_tokens <= MAX_TOKENS:
            current["units"] += units
            current["tokens"] += para_tokens
            continue
        for unit in units:                                        # paragraph longer than a chunk: fill by sentence
            if current["tokens"] + unit[1] > MAX_TOKENS:
                carried = overlap_tail(current["units"])
                emit(); current = start(para["heading"], carried)
            current["units"].append(unit)
            current["tokens"] += unit[1]
    emit()

    # Merge a tiny final chunk into the previous one if it fits.
    if len(chunks) >= 2 and chunks[-1]["tokens"] < MIN_TOKENS \
            and chunks[-2]["tokens"] + chunks[-1]["tokens"] <= MAX_TOKENS:
        last = chunks.pop()
        chunks[-1]["units"] += last["units"]
        chunks[-1]["tokens"] += last["tokens"]
    return [{"heading": c["heading"], "text": " ".join(u[0] for u in c["units"])} for c in chunks]

# Quick look at one document before running everything.
example = chunk_paragraphs(cleaned["tem_bulletin_2024_08"])
print(len(example), "chunks; tokens:", count_tokens([c["text"] for c in example]))
print("\nheadings:", [c["heading"] for c in example])
print("\nchunk 0:\n", example[0]["text"][:600])

## Build all chunks

Each chunk becomes a record with everything a citation and the index need. `embed_text` adds the document title and section heading so a passage is understandable on its own.

In [ ]:
records = []
for d in docs.itertuples():
    chunks = chunk_paragraphs(cleaned[d.doc_id])
    for i, c in enumerate(chunks):
        prefix = f"{d.title}. " + (f"{c['heading'].rstrip(':')}. " if c["heading"] else "")
        records.append({
            "chunk_id": f"{d.doc_id}#{i:03d}", "doc_id": d.doc_id, "chunk_index": i,
            "source_id": d.source_id, "source_type": d.source_type, "title": d.title,
            "language": d.language, "published": d.published, "published_effective": d.published_effective,
            "landing_url": d.landing_url, "licence": d.licence, "heading": c["heading"],
            "text": c["text"], "embed_text": prefix + c["text"],
        })

frame = pd.DataFrame(records)
frame["n_tokens"] = count_tokens(frame.text.tolist())
frame["n_tokens_embed"] = count_tokens(frame.embed_text.tolist())
print("chunks:", len(frame), "| documents with chunks:", frame.doc_id.nunique(), "of", len(docs))
print(frame.groupby("source_id").agg(chunks=("chunk_id", "size"), tokens_median=("n_tokens", "median"),
                                     tokens_max=("n_tokens", "max")))
print("\nchunks per language:", frame.language.value_counts().to_dict())
print("chunks per heading:", frame.heading.fillna("(no heading)").value_counts().head(8).to_dict())

## Checks

The build stops if any of these fail.

In [ ]:
assert frame.chunk_id.is_unique, "duplicate chunk ids"
assert (frame.text.str.strip().str.len() > 0).all(), "empty chunk"
assert frame.n_tokens.max() <= MAX_TOKENS + 8, f"chunk too long: {frame.n_tokens.max()} tokens"
assert frame.doc_id.nunique() == len(docs), "a document produced no chunks"
assert not frame.text.str.contains("­|\xa0").any(), "soft hyphen or non-breaking space left in text"
assert frame.published_effective.notna().all()

small = frame[frame.n_tokens < MIN_TOKENS]
print(f"chunks under the {MIN_TOKENS}-token minimum: {len(small)} ({100 * len(small) / len(frame):.1f}%)"
      " (a short final section can legitimately be small)")
print("token length: median", int(frame.n_tokens.median()), "| 90th pct", int(frame.n_tokens.quantile(.9)),
      "| max", int(frame.n_tokens.max()))
print("with the title/heading prefix, longest embed_text:", int(frame.n_tokens_embed.max()), "tokens (bge-m3 accepts 8,192)")
print("all checks passed")

## Read a few chunks

A last human check. Sample chunks from an older and a newer document, in both languages.

In [ ]:
sample_ids = ["tem_bulletin_2013_06", "tem_bulletin_2019_05", "keha_bulletin_20260821118810"]
for doc_id in sample_ids:
    rows = frame[(frame.doc_id == doc_id) & frame.heading.fillna("").str.contains("vacan|avoim|Jobs", case=False)]
    if rows.empty:
        rows = frame[frame.doc_id == doc_id]
    r = rows.iloc[0]
    print(f"--- {r.chunk_id} | {r.language} | effective {r.published_effective} | {r.n_tokens} tokens | heading: {r.heading}")
    print(r.text[:500], "\n")

## Write outputs

The chunk file is git-ignored (rebuilt by this notebook); the manifest is committed. The manifest links the chunks to the exact corpus manifest and records the parameters, so a later notebook can refuse to use chunks made from a different corpus.

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

NOW_UTC = dt.datetime.now(dt.timezone.utc)
TS = NOW_UTC.strftime("%Y%m%dT%H%M%SZ")
chunks_path = OUT_DIR / "chunks.jsonl"
frame.drop(columns=["n_tokens_embed"]).assign(n_tokens=frame.n_tokens).to_json(
    chunks_path, orient="records", lines=True, force_ascii=False)

per_doc = report.merge(frame.groupby("doc_id").size().rename("chunks").reset_index(), on="doc_id", how="left")
per_doc["text_sha256"] = per_doc.doc_id.map(lambda d: sha256_file(TEXT_DIR / f"{d}.txt"))

manifest = {
    "name": "JobAI RAG chunks v1",
    "created_utc": NOW_UTC.isoformat(),
    "corpus_manifest": {"path": str(manifest_path.relative_to(REPO)), "sha256": sha256_file(manifest_path)},
    "parameters": {"chunk_size_tokens": MAX_TOKENS, "chunk_overlap_tokens": OVERLAP_TOKENS,
                   "min_chunk_tokens": MIN_TOKENS, "tokenizer": TOKENIZER_ID, "splitter": CHUNK_CFG["splitter"]},
    "counts": {"documents": int(len(docs)), "chunks": int(len(frame)),
               "chunks_by_source": frame.groupby("source_id").size().to_dict(),
               "chunks_by_language": frame.language.value_counts().to_dict(),
               "tokens_median": int(frame.n_tokens.median()), "tokens_max": int(frame.n_tokens.max())},
    "documents_without_vacancy_paragraph": no_vacancy.doc_id.tolist(),
    "dropped_on_purpose": ["tables", "chart and map labels", "contacts, links and referencing instructions"],
    "outputs": {"chunks": {"path": str(chunks_path.relative_to(REPO)), "rows": int(len(frame)),
                           "sha256": sha256_file(chunks_path)}},
    "documents": per_doc.to_dict(orient="records"),
    "packages": {n: importlib.metadata.version(n) for n in ["tokenizers", "pandas"]},
}
manifest_out = MAN / f"rag_chunks_{TS}.json"
manifest_out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
print("wrote:", chunks_path)
print("wrote:", manifest_out)

## Known limitations (as of 2026-09-21)

- **Tables are not searchable** (see above). Numbers must come from PxWeb.
- **Section labels are coarse.** Only the recurring main headings are recognised; sub-sections inside them share the label. Chunks under a heading are still separated by paragraph.
- **The 2013–2014 bulletins are short summaries** (about 4 pages), so they yield few chunks each.
- **Page numbers are not kept.** The extraction joined pages, so a citation points to the document and section, not the page.
- **Sentence splitting is simple** (a full stop followed by a capital letter). An abbreviation such as `n. 5` can cause an occasional early split; this only moves a chunk boundary.
- **No language is translated.** Finnish chunks stay Finnish; cross-language retrieval is `bge-m3`'s job (notebook 11) and is what the evaluator should test first.
- **One document has no vacancy paragraph** (the October 2025 bulletin shows vacancies only as a chart and a table). It is listed in the manifest.

**Next:** notebook 11 embeds `embed_text` with `BAAI/bge-m3`, builds the ChromaDB index and the BM25 keyword index, and writes an index manifest that points back to this notebook's chunk manifest.